# PDCP · N04 — Inference pilot (7 open-weight VLMs × 100 pairs)
**Notebook 4 of 13 — first GPU notebook.** Kaggle: Accelerator **GPU T4 x2** · Internet **On** · **Save & Run All (Commit)**. Expected ≈ 1–1.5 GPU-hours.

**Inputs:** N03 output dataset (`pairs.parquet`, `drawings.parquet`, `images/768/`, `pdcp_utils.py`).
**Secret (optional):** `HF_TOKEN` in *Add-ons → Secrets* — needed only for Gemma 3 (gated). Without it Gemma is skipped automatically.

**What it does**
1. Builds the **primary balanced analysis set** (Δ 0–0.15 full + Δ 0.20 subsampled to equal counts per condition) and re-checks decorrelation.
2. Writes the inference worker `pdcp_infer.py` (reused by every later GPU notebook).
3. Runs each model in its **own process**, two at a time (one per T4), on the same 100 stratified pairs with prompt **P-Def**, greedy decoding. Each process: load → fp16 sanity check (auto-retry bf16) → 100 queries → batch-4 throughput test.
4. Reports load success, seconds/query, parse rate, pilot accuracy, VRAM, and **projects GPU-hours for E1/E2/E3/E4** → `run_plan.json` used by N05+.

**Outputs** (`/kaggle/working/n04_output/`): `pdcp_infer.py`, `pdcp_utils.py`, `analysis_sets.parquet`, `pilot_pairs.csv`, `results/pilot/*.jsonl`, `pilot_summary.csv`, `run_plan.json`, `logs/*.log`, figures, `MANIFEST.txt`.

In [1]:
# ==== SETTINGS ====
INPUT_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n3-output"   # searched recursively for pairs.parquet
OUTPUT_DIR = "/kaggle/working/n04_output"
MODELS = ["Q3VL-8B", "Q3VL-2B", "Q25VL-7B", "SmolVLM-2B", "Q3VL-4B", "InternVL-4B", "Gemma3-4B"]   # big+small interleaved
N_PILOT = 100
BATCH_TEST = 4
PROMPT = "P-Def"
MAX_NEW_TOKENS = 160
PER_MODEL_TIMEOUT_MIN = 45
SEED = 2026

In [2]:
import os, sys, json, time, glob, shutil, subprocess, importlib
os.makedirs(f"{OUTPUT_DIR}/results/pilot", exist_ok=True); os.makedirs(f"{OUTPUT_DIR}/logs", exist_ok=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
import torch
N_GPU = torch.cuda.device_count(); print("GPUs:", N_GPU)
assert N_GPU >= 1, "No GPU: set Accelerator = GPU T4 x2"
# packages for the worker processes (fresh interpreters pick up the upgrade)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.57", "accelerate", "bitsandbytes",
                    "qwen-vl-utils", "num2words", "pyarrow"], capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-1500:])
print(subprocess.run([sys.executable, "-c", "import transformers, accelerate, bitsandbytes; print('transformers', transformers.__version__, '| bnb', bitsandbytes.__version__)"],
                     capture_output=True, text=True).stdout)
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN found")
except Exception as e:
    print("No HF_TOKEN secret -> gated models (Gemma) will be skipped:", type(e).__name__)
if HF_TOKEN is None and "Gemma3-4B" in MODELS:
    MODELS = [m for m in MODELS if m != "Gemma3-4B"]

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

GPUs: 2
 ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.

transformers 5.23.0 | bnb 0.50.2

No HF_TOKEN secret -> gated models (Gemma) will be skipped: BackendError


## 1. Inputs and the primary balanced analysis set

In [3]:
import numpy as np, pandas as pd
from scipy.stats import pearsonr, spearmanr
pp = sorted(glob.glob(f"{INPUT_ROOT}/**/pairs.parquet", recursive=True))
assert pp, "pairs.parquet not found - attach the N03 output dataset"
N03 = os.path.dirname(pp[0]); print("N03 dir:", N03)
PR = pd.read_parquet(f"{N03}/pairs.parquet"); DR = pd.read_parquet(f"{N03}/drawings.parquet")
shutil.copy(f"{N03}/pdcp_utils.py", f"{OUTPUT_DIR}/pdcp_utils.py")
png = dict(zip(DR.drawing_id, DR.png))
missing = [d for d in set(PR.drawing_A) | set(PR.drawing_B) if not os.path.exists(f"{N03}/{png[d]}")]
print(PR.set.value_counts().to_dict(), "| missing PNGs:", len(missing))
assert not missing

main = PR[PR.set == "main"].copy()
cnt = main.groupby(["dstress_design", "cond"]).size().unstack()
print(cnt)
rs = np.random.default_rng(7)
keep = []
for (d, c), g in main.groupby(["dstress_design", "cond"]):
    k = int(cnt.loc[d].min()) if d == 0.20 else len(g)       # balance the thin Δ=0.20 row
    keep += list(g.sample(n=k, random_state=int(rs.integers(1e6))).index)
main["primary"] = main.index.isin(keep)
def corr(df):
    return {p: {"pearson": round(float(pearsonr(df.d_ksm, df["d_" + p])[0]), 4), "spearman": round(float(spearmanr(df.d_ksm, df["d_" + p])[0]), 4)}
            for p in ["nu", "eld", "xr"]}
CORR = {"all_main": corr(main), "primary": corr(main[main.primary])}
print("primary set:", int(main.primary.sum()), "pairs |", json.dumps(CORR["primary"]))
AS = PR[["pair_id", "set"]].merge(main[["pair_id", "primary"]], on="pair_id", how="left")
AS["primary"] = AS.primary.fillna(False).astype(bool)
AS.to_parquet(f"{OUTPUT_DIR}/analysis_sets.parquet", index=False)

N03 dir: /kaggle/input/datasets/resadussalehinrafsan/kbs-n3-output/n03_output
{'counterfactual': 2839, 'main': 1926, 'e1b_mooney': 289, 'fewshot': 30} | missing PNGs: 0
cond            conflict_all  conflict_eld  conflict_nu  conflict_xr  \
dstress_design                                                         
0.00                      60            60           60           60   
0.03                      60            60           60           60   
0.06                      60            60           60           60   
0.10                      60            60           60           60   
0.15                      53            60           59           58   
0.20                      13            24           13           13   

cond            congruent  neutral  
dstress_design                      
0.00                   58       60  
0.03                   59       60  
0.06                   60       60  
0.10                   60       60  
0.15                   60       

/tmp/ipykernel_23/2247518779.py:28: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  AS["primary"] = AS.primary.fillna(False).astype(bool)


## 2. Pilot sample (100 stratified pairs, default image order)

In [4]:
prim = main[main.primary]
pil = prim.groupby(["dstress_design", "cond"], group_keys=False).sample(n=3, random_state=SEED, replace=False)
pil = pil.sample(n=min(N_PILOT, len(pil)), random_state=SEED)
def to_rows(df, orders=None):
    rows = []
    for r in df.itertuples():
        o = orders or [r.default_order]
        for order in o:
            a, b = f"{N03}/{png[r.drawing_A]}", f"{N03}/{png[r.drawing_B]}"
            img1, img2 = (a, b) if order == "AB" else (b, a)
            corr_img = "same" if r.correct == "same" else ("1" if (r.correct == "A") == (order == "AB") else "2")
            rows.append({"pair_id": r.pair_id, "order": order, "img1": img1, "img2": img2, "correct_img": corr_img,
                         "dstress": r.dstress_design, "cond": r.cond, "family": f"{r.family}{r.n}"})
    return pd.DataFrame(rows)
PIL = to_rows(pil)
PIL.to_csv(f"{OUTPUT_DIR}/pilot_pairs.csv", index=False)
print(len(PIL), "pilot queries | correct:", PIL.correct_img.value_counts().to_dict())

100 pilot queries | correct: {'2': 44, '1': 40, 'same': 16}


## 3. Inference worker (saved for all later GPU notebooks)

In [5]:
%%writefile /kaggle/working/n04_output/pdcp_infer.py
"""
pdcp_infer.py — PDCP VLM inference worker (v0.2.0). One model per process.

Usage:
  python pdcp_infer.py --model Q3VL-4B --pairs pairs.csv --out results.jsonl --summary sum.json
                       [--prompt P-Def] [--batch 1] [--batch_test 4] [--max_new 160] [--hf_token TOKEN]

pairs.csv columns: pair_id, order (AB|BA), img1, img2 (absolute PNG paths), correct_img ("1"|"2"|"same")
Resumable: rows already in --out (same pair_id+order+prompt) are skipped.
"""
import argparse, gc, json, os, re, sys, time, traceback

MODELS = {
    "Q3VL-2B":     {"ids": ["Qwen/Qwen3-VL-2B-Instruct"], "quant": None},
    "Q3VL-4B":     {"ids": ["Qwen/Qwen3-VL-4B-Instruct"], "quant": None},
    "Q3VL-8B":     {"ids": ["Qwen/Qwen3-VL-8B-Instruct"], "quant": "nf4"},
    "Q25VL-7B":    {"ids": ["Qwen/Qwen2.5-VL-7B-Instruct"], "quant": "nf4"},
    "Gemma3-4B":   {"ids": ["google/gemma-3-4b-it"], "quant": None},
    "InternVL-4B": {"ids": ["OpenGVLab/InternVL3_5-4B-HF", "OpenGVLab/InternVL3-2B-hf"], "quant": None},
    "SmolVLM-2B":  {"ids": ["HuggingFaceTB/SmolVLM2-2.2B-Instruct"], "quant": None},
}

SYSTEM = ("You are a careful visual analyst. You will be shown two images of node-link diagrams that depict the SAME "
          "network (same nodes and same edges), drawn with different node positions. Answer the question strictly in "
          "the JSON format requested. Do not add any text outside the JSON.")
DEFINITION = ("In a node-link diagram, each circle is a node and each line is an edge. The \"stress\" of a drawing measures "
              "how well the straight-line distances between pairs of nodes match the graph-theoretic distances between them "
              "(the number of edges on the shortest path connecting them). A drawing has LOW stress when pairs of nodes that "
              "are few edges apart are drawn close together and pairs that are many edges apart are drawn far apart, "
              "consistently across the whole drawing. A drawing has HIGH stress when these distances disagree.")
JSON_SPEC = ('Respond with JSON only:\n{"answer": "1" | "2" | "same", "confidence": <integer 1-5>, '
             '"reason": "<at most 60 words>"}')
PROMPTS = {
    "P-Def": DEFINITION + "\n\nImage 1 and Image 2 show the same network. Which image has LOWER stress?\n\n" + JSON_SPEC,
    "P-Expert": ("You are an expert in graph drawing and network visualization. Image 1 and Image 2 show the same network. "
                 "Which image has lower stress?\n\n" + JSON_SPEC),
}


def parse_answer(text):
    out = {"answer": None, "confidence": None, "reason": None, "parse_ok": False}
    if not text:
        return out
    m = re.search(r"\{.*\}", text, re.S)
    if m:
        try:
            obj = json.loads(m.group(0))
            a = str(obj.get("answer", "")).strip().lower().replace("image", "").strip()
            if a in ("1", "2"):
                out["answer"] = a
            elif a in ("same", "no clear difference"):
                out["answer"] = "same"
            c = obj.get("confidence")
            out["confidence"] = int(c) if c is not None and str(c).strip().isdigit() else None
            out["reason"] = obj.get("reason")
            out["parse_ok"] = out["answer"] is not None
            if out["parse_ok"]:
                return out
        except Exception:
            pass
    m = re.search(r'answer"?\s*(?:is)?\s*[:=]?\s*"?\s*(image\s*)?(1|2|same|no clear difference)\b', text, re.I)
    if m:
        a = m.group(2).lower()
        out["answer"] = "same" if a.startswith(("same", "no")) else a
        out["parse_ok"] = True
        r = re.search(r'"reason"\s*:\s*"([^"]*)', text)
        out["reason"] = r.group(1) if r else None
    return out


def build_messages(img1, img2, prompt_id, with_images=True):
    def im(x):
        return {"type": "image", "image": x} if with_images else {"type": "image"}
    return [{"role": "user", "content": [
        {"type": "text", "text": SYSTEM + "\n\nImage 1:"}, im(img1),
        {"type": "text", "text": "Image 2:"}, im(img2),
        {"type": "text", "text": PROMPTS[prompt_id]}]}]


class Runner:
    def __init__(self, key, hf_token=None, log=print):
        import torch
        from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
        self.torch, self.key, self.log = torch, key, log
        spec = MODELS[key]
        self.route, self.model, self.proc, self.model_id, self.dtype_name = None, None, None, None, None
        errors = []
        for mid in spec["ids"]:
            for dt in (["float16", "bfloat16"] if spec["quant"] is None else ["float16"]):
                try:
                    t0 = time.time()
                    kw = {"device_map": {"": 0}, "token": hf_token, "low_cpu_mem_usage": True}
                    if spec["quant"] == "nf4":
                        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                                        bnb_4bit_compute_dtype=torch.float16,
                                                                        bnb_4bit_use_double_quant=True)
                    else:
                        kw["dtype"] = getattr(torch, dt)
                    if self.model is None or self.model_id != mid:
                        self._free()
                        self.model = AutoModelForImageTextToText.from_pretrained(mid, **kw).eval()
                        self.proc = AutoProcessor.from_pretrained(mid, token=hf_token)
                    self.model_id, self.dtype_name, self.load_s = mid, (spec["quant"] or dt), time.time() - t0
                    return
                except Exception as e:
                    errors.append(f"{mid} {dt}: {type(e).__name__}: {str(e)[:300]}")
                    self.log("load failed: " + errors[-1])
                    self._free()
        raise RuntimeError("all loads failed | " + " || ".join(errors))

    def reload_dtype(self, dt):
        """Reload the same model in another dtype (used when fp16 produces garbage)."""
        from transformers import AutoModelForImageTextToText
        self._free(keep_proc=True)
        self.model = AutoModelForImageTextToText.from_pretrained(self.model_id, device_map={"": 0},
                                                                 dtype=getattr(self.torch, dt), low_cpu_mem_usage=True).eval()
        self.dtype_name = dt

    def _free(self, keep_proc=False):
        if self.model is not None:
            del self.model
            self.model = None
        if not keep_proc:
            self.proc = None
        gc.collect()
        try:
            self.torch.cuda.empty_cache()
        except Exception:
            pass

    def _inputs(self, batch, prompt_id):
        """batch = list of (img1, img2). Tries route A (chat template loads images) then B (processor call)."""
        torch = self.torch
        routes = [self.route] if self.route else ["A", "B_nested", "B_flat"]
        last = None
        for route in routes:
            try:
                if route == "A":
                    msgs = [build_messages(a, b, prompt_id) for a, b in batch]
                    if len(batch) > 1:
                        self.proc.tokenizer.padding_side = "left"
                    inp = self.proc.apply_chat_template(msgs if len(batch) > 1 else msgs[0], add_generation_prompt=True,
                                                        tokenize=True, return_dict=True, return_tensors="pt",
                                                        padding=len(batch) > 1)
                else:
                    texts = [self.proc.apply_chat_template(build_messages(a, b, prompt_id, with_images=False),
                                                           add_generation_prompt=True, tokenize=False) for a, b in batch]
                    imgs = [[a, b] for a, b in batch] if route == "B_nested" else [x for ab in batch for x in ab]
                    if len(batch) > 1:
                        self.proc.tokenizer.padding_side = "left"
                    inp = self.proc(text=texts, images=imgs, return_tensors="pt", padding=True)
                self.route = route
                dev = self.model.device
                mdt = torch.float16 if self.dtype_name in ("nf4", "float16") else getattr(torch, self.dtype_name)
                return {k: (v.to(dev, dtype=mdt) if torch.is_floating_point(v) else v.to(dev)) if hasattr(v, "to") else v
                        for k, v in inp.items()}
            except Exception as e:
                last = e
                if self.route:
                    raise
        raise RuntimeError(f"no input route worked: {type(last).__name__}: {last}")

    def generate(self, batch, prompt_id, max_new=160):
        torch = self.torch
        inp = self._inputs(batch, prompt_id)
        n_in = inp["input_ids"].shape[1]
        with torch.inference_mode():
            out = self.model.generate(**inp, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None, top_k=None)
        texts = self.proc.batch_decode(out[:, n_in:], skip_special_tokens=True)
        return texts, int(n_in)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True); ap.add_argument("--pairs", required=True)
    ap.add_argument("--out", required=True); ap.add_argument("--summary", required=True)
    ap.add_argument("--prompt", default="P-Def"); ap.add_argument("--batch", type=int, default=1)
    ap.add_argument("--batch_test", type=int, default=0); ap.add_argument("--max_new", type=int, default=160)
    ap.add_argument("--hf_token", default=None); ap.add_argument("--limit", type=int, default=0)
    a = ap.parse_args()
    import pandas as pd, torch
    from PIL import Image
    def log(s): print(f"[{a.model} {time.strftime('%H:%M:%S')}] {s}", flush=True)
    summ = {"model": a.model, "prompt": a.prompt, "ok": False, "error": None}
    t_start = time.time()
    try:
        P = pd.read_csv(a.pairs)
        if a.limit: P = P.head(a.limit)
        done = set()
        if os.path.exists(a.out):
            for line in open(a.out):
                try:
                    r = json.loads(line); done.add((r["pair_id"], r["order"], r["prompt_id"]))
                except Exception:
                    pass
        todo = [r for r in P.to_dict("records") if (r["pair_id"], r["order"], a.prompt) not in done]
        log(f"{len(P)} pairs, {len(todo)} to do")
        torch.cuda.reset_peak_memory_stats()
        R = Runner(a.model, a.hf_token, log)
        log(f"loaded {R.model_id} as {R.dtype_name} in {R.load_s:.0f}s")
        summ.update({"model_id": R.model_id, "dtype": R.dtype_name, "load_s": round(R.load_s, 1)})
        load = lambda p: Image.open(p).convert("RGB")

        # warm-up + dtype sanity: if fp16 gives zero parseable answers on 3 items, retry in bf16
        if todo:
            w = todo[:3]
            texts, _ = R.generate([(load(r["img1"]), load(r["img2"])) for r in w[:1]], a.prompt, a.max_new)
            ok = parse_answer(texts[0])["parse_ok"]
            if not ok:
                t2 = [R.generate([(load(r["img1"]), load(r["img2"]))], a.prompt, a.max_new)[0][0] for r in w[1:]]
                if not any(parse_answer(t)["parse_ok"] for t in t2) and R.dtype_name == "float16":
                    log("fp16 output unparseable on warm-up -> retrying in bfloat16. sample: " + repr(texts[0][:200]))
                    try:
                        R.reload_dtype("bfloat16"); summ["dtype"] = "bfloat16"
                    except Exception as e:
                        log(f"bf16 reload failed ({type(e).__name__}); reloading fp16 and continuing")
                        R.reload_dtype("float16"); summ["dtype_note"] = "fp16 unparseable, bf16 unavailable"
        summ["route"] = R.route

        f = open(a.out, "a")
        times, n_ok, n_corr, n = [], 0, 0, 0
        bs = max(1, a.batch)
        for i in range(0, len(todo), bs):
            chunk = todo[i:i + bs]
            t0 = time.time()
            try:
                texts, n_in = R.generate([(load(r["img1"]), load(r["img2"])) for r in chunk], a.prompt, a.max_new)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if bs > 1:
                    log("OOM at batch %d -> falling back to batch 1" % bs); bs = 1
                    texts, n_in = [], 0
                    for r in chunk:
                        t, n_in = R.generate([(load(r["img1"]), load(r["img2"]))], a.prompt, a.max_new); texts += t
                else:
                    raise
            dt = (time.time() - t0) / len(chunk)
            for r, t in zip(chunk, texts):
                p = parse_answer(t)
                corr = p["answer"] == str(r["correct_img"]) if p["parse_ok"] else None
                rec = {"pair_id": r["pair_id"], "order": r["order"], "model": a.model, "model_id": R.model_id,
                       "dtype": R.dtype_name, "prompt_id": a.prompt, "raw_text": t, "parsed": p, "parse_ok": p["parse_ok"],
                       "correct_img": str(r["correct_img"]), "correct": corr, "latency_s": round(dt, 3), "input_tokens": n_in,
                       "batch": len(chunk), "ts": time.strftime("%Y-%m-%dT%H:%M:%S")}
                f.write(json.dumps(rec) + "\n")
                n += 1; n_ok += p["parse_ok"]; n_corr += bool(corr)
            f.flush(); times.append(dt)
            if n % 50 < len(chunk):
                log(f"{n}/{len(todo)} | {dt:.2f}s/q | parse {n_ok/n:.2f} | acc {n_corr/max(1,n_ok):.2f}")
        f.close()
        summ.update({"n": n, "sec_per_query": round(float(sum(times) / max(1, len(times))), 3), "parse_rate": round(n_ok / max(1, n), 3),
                     "acc_on_parsed": round(n_corr / max(1, n_ok), 3), "input_tokens": n_in if todo else None})

        # optional batched-throughput test on the first items (not written to results)
        if a.batch_test and a.batch_test > 1 and len(P) >= a.batch_test * 3:
            rows = P.to_dict("records")[: a.batch_test * 3]
            try:
                t0 = time.time()
                for i in range(0, len(rows), a.batch_test):
                    texts, _ = R.generate([(load(r["img1"]), load(r["img2"])) for r in rows[i:i + a.batch_test]], a.prompt, a.max_new)
                summ["batch_test"] = {"batch": a.batch_test, "sec_per_query": round((time.time() - t0) / len(rows), 3),
                                      "parse_rate": round(sum(parse_answer(t)["parse_ok"] for t in texts) / len(texts), 3)}
            except Exception as e:
                summ["batch_test"] = {"batch": a.batch_test, "error": f"{type(e).__name__}: {str(e)[:200]}"}
            log(f"batch test: {summ['batch_test']}")
        summ["vram_peak_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)
        summ["ok"] = True
    except Exception as e:
        summ["error"] = f"{type(e).__name__}: {str(e)[:1500]}"
        log("FAILED: " + summ["error"]); traceback.print_exc()
    summ["wall_s"] = round(time.time() - t_start, 1)
    json.dump(summ, open(a.summary, "w"), indent=2)
    log(f"done: {json.dumps({k: summ.get(k) for k in ['ok', 'n', 'sec_per_query', 'parse_rate', 'acc_on_parsed', 'vram_peak_gb']})}")


if __name__ == "__main__":
    main()


Writing /kaggle/working/n04_output/pdcp_infer.py


## 4. Run all models (two processes in parallel, one per GPU)

In [6]:
T0 = time.time()
HF_CACHE = "/tmp/hf_cache"
queue = list(MODELS); running = {}   # gpu -> (model, proc, start, logf)
def cache_dir_for(model):
    import pdcp_infer as I
    return [f"{HF_CACHE}/models--{mid.replace('/', '--')}" for mid in I.MODELS[model]["ids"]]
sys.path.insert(0, OUTPUT_DIR); import pdcp_infer as I; importlib.reload(I)
def launch(model, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), HF_HUB_CACHE=HF_CACHE, HF_HUB_ENABLE_HF_TRANSFER="0",
               TOKENIZERS_PARALLELISM="false", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    cmd = [sys.executable, f"{OUTPUT_DIR}/pdcp_infer.py", "--model", model, "--pairs", f"{OUTPUT_DIR}/pilot_pairs.csv",
           "--out", f"{OUTPUT_DIR}/results/pilot/{model}.jsonl", "--summary", f"{OUTPUT_DIR}/results/pilot/{model}.summary.json",
           "--prompt", PROMPT, "--batch", "1", "--batch_test", str(BATCH_TEST), "--max_new", str(MAX_NEW_TOKENS)]
    if HF_TOKEN: cmd += ["--hf_token", HF_TOKEN]
    logf = open(f"{OUTPUT_DIR}/logs/{model}.log", "w")
    p = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=env)
    running[gpu] = (model, p, time.time(), logf)
    print(f"[{(time.time()-T0)/60:5.1f} min] start {model} on GPU {gpu}")
while queue or running:
    for gpu in range(N_GPU):
        if gpu not in running and queue:
            launch(queue.pop(0), gpu)
    time.sleep(20)
    for gpu, (model, p, st, logf) in list(running.items()):
        timeout = time.time() - st > PER_MODEL_TIMEOUT_MIN * 60
        if p.poll() is not None or timeout:
            if timeout and p.poll() is None:
                p.kill(); print(f"   {model} killed after {PER_MODEL_TIMEOUT_MIN} min (results so far kept)")
            logf.close(); del running[gpu]
            tail = open(f"{OUTPUT_DIR}/logs/{model}.log").read().strip().splitlines()[-2:]
            print(f"[{(time.time()-T0)/60:5.1f} min] end {model}: " + " | ".join(tail)[-400:])
            for d in cache_dir_for(model): shutil.rmtree(d, ignore_errors=True)   # free disk for the next model
print(f"all models finished in {(time.time()-T0)/60:.1f} min")

[  0.0 min] start Q3VL-8B on GPU 0
[  0.0 min] start Q3VL-2B on GPU 1
[ 12.0 min] end Q3VL-2B: [Q3VL-2B 19:19:20] batch test: {'batch': 4, 'sec_per_query': 2.874, 'parse_rate': 1.0} | [Q3VL-2B 19:19:20] done: {"ok": true, "n": 100, "sec_per_query": 5.096, "parse_rate": 1.0, "acc_on_parsed": 0.36, "vram_peak_gb": 6.59}
[ 12.0 min] start Q25VL-7B on GPU 1
[ 28.7 min] end Q25VL-7B: [Q25VL-7B 19:36:16] batch test: {'batch': 4, 'sec_per_query': 6.756, 'parse_rate': 1.0} | [Q25VL-7B 19:36:16] done: {"ok": true, "n": 100, "sec_per_query": 7.634, "parse_rate": 1.0, "acc_on_parsed": 0.29, "vram_peak_gb": 10.06}
[ 28.7 min] start SmolVLM-2B on GPU 1
   Q3VL-8B killed after 45 min (results so far kept)
[ 45.0 min] end Q3VL-8B: [transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs` | [transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[ 45.0 min] start Q3VL-4B on GPU 0
[ 57.3 min] end Q

## 5. Summary, accuracy and GPU-hour projection

In [7]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
S, R = [], []
for m in MODELS:
    sf = f"{OUTPUT_DIR}/results/pilot/{m}.summary.json"
    s = json.load(open(sf)) if os.path.exists(sf) else {"model": m, "ok": False, "error": "no summary (killed or crashed)"}
    S.append(s)
    jf = f"{OUTPUT_DIR}/results/pilot/{m}.jsonl"
    if os.path.exists(jf):
        R += [json.loads(l) for l in open(jf)]
S = pd.DataFrame(S); R = pd.DataFrame(R)
if len(R):
    R = R.merge(PIL[["pair_id", "order", "dstress", "cond", "family"]], on=["pair_id", "order"], how="left")
    R["ans"] = R.parsed.map(lambda p: p.get("answer"))
    extra = R.groupby("model").agg(answer_same=("ans", lambda x: float((x == "same").mean())),
                                   answer_1=("ans", lambda x: float((x == "1").mean())),
                                   mean_conf=("parsed", lambda x: float(np.nanmean([p.get("confidence") or np.nan for p in x]))))
    S = S.merge(extra, left_on="model", right_index=True, how="left")
def best_spq(s):
    bt = s.get("batch_test") if isinstance(s.get("batch_test"), dict) else {}
    if bt.get("sec_per_query") and bt.get("parse_rate", 0) >= (s.get("parse_rate") or 0) - 0.05:
        return bt["sec_per_query"], bt["batch"]
    return s.get("sec_per_query"), 1
n_main = int((PR.set == "main").sum()); n_cf = int((PR.set == "counterfactual").sum())
n_e1b = int((PR.set == "e1b_mooney").sum())
plan = {"prompt": PROMPT, "models": {}, "n_gpu": N_GPU}
for s in S.to_dict("records"):
    if not s.get("ok"): plan["models"][s["model"]] = {"usable": False, "error": s.get("error")}; continue
    spq, bs = best_spq(s)
    load_h = (s.get("load_s") or 300) / 3600 + 0.08
    e1 = (2 * n_main + n_e1b) * spq / 3600 + load_h; e2 = n_cf * spq / 3600 + load_h
    e3 = 400 * 12 * spq / 3600 + load_h; e4 = 400 * 10 * spq / 3600 + load_h
    plan["models"][s["model"]] = {"usable": s.get("parse_rate", 0) >= 0.8, "model_id": s.get("model_id"), "dtype": s.get("dtype"),
                                  "route": s.get("route"), "batch": bs, "sec_per_query": spq, "parse_rate": s.get("parse_rate"),
                                  "pilot_acc": s.get("acc_on_parsed"), "vram_gb": s.get("vram_peak_gb"),
                                  "gpu_hours": {"E1+E1b": round(e1, 2), "E2": round(e2, 2), "E3": round(e3, 2), "E4": round(e4, 2)}}
U = {k: v for k, v in plan["models"].items() if v.get("usable")}
tot = {e: round(sum(v["gpu_hours"][e] for v in U.values()), 1) for e in ["E1+E1b", "E2"]}
plan["totals_all_usable_models_gpu_hours"] = tot
plan["note"] = "GPU-hours per process; with 2 T4s running two processes in parallel the Kaggle quota used is about half of the summed hours."
plan["kaggle_quota_hours_estimate_E1_E2"] = round((tot["E1+E1b"] + tot["E2"]) / max(1, N_GPU), 1)
ab = sorted(U, key=lambda k: U[k]["sec_per_query"])
plan["suggested_ablation_models"] = [m for m in ["Q3VL-4B", "Gemma3-4B", "InternVL-4B"] if m in U] or ab[:3]
json.dump(plan, open(f"{OUTPUT_DIR}/run_plan.json", "w"), indent=2, default=str)
cols = ["model", "ok", "model_id", "dtype", "route", "load_s", "sec_per_query", "batch_test", "parse_rate", "acc_on_parsed",
        "answer_same", "answer_1", "mean_conf", "vram_peak_gb", "wall_s", "error"]
S[[c for c in cols if c in S]].to_csv(f"{OUTPUT_DIR}/pilot_summary.csv", index=False)
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 60)
print(S[[c for c in cols if c in S and c not in ("batch_test", "error")]].round(3).to_string(index=False))
print("\nerrors:", S[S.ok != True][["model", "error"]].to_string(index=False) if (S.ok != True).any() else "none")
print("\nprojected GPU-hours (sum over usable models):", tot, "| quota with 2 GPUs ≈", plan["kaggle_quota_hours_estimate_E1_E2"], "h for E1+E2")

      model    ok                             model_id    dtype route  load_s  sec_per_query  parse_rate  acc_on_parsed  answer_same  answer_1  mean_conf  vram_peak_gb  wall_s
    Q3VL-8B False                                  NaN      NaN   NaN     NaN            NaN         NaN            NaN          0.0     0.087        5.0           NaN     NaN
    Q3VL-2B  True            Qwen/Qwen3-VL-2B-Instruct  float16     A   129.2          5.096        1.00           0.36          0.0     0.880        5.0          6.59   694.1
   Q25VL-7B  True          Qwen/Qwen2.5-VL-7B-Instruct      nf4     A   133.7          7.634        1.00           0.29          0.0     0.380        4.0         10.06   989.6
 SmolVLM-2B  True HuggingFaceTB/SmolVLM2-2.2B-Instruct bfloat16     A    58.1         18.845        0.08           0.25          0.0     0.080        5.0         14.03  1961.8
    Q3VL-4B  True            Qwen/Qwen3-VL-4B-Instruct  float16     A    77.8          5.904        1.00           0.29 

In [8]:
if len(R):
    ok = R[R.parse_ok]
    fig, ax = plt.subplots(1, 2, figsize=(15, 4.2))
    acc_d = ok.groupby(["model", "dstress"]).correct.mean().unstack(0)
    acc_d.plot(ax=ax[0], marker="o"); ax[0].axhline(1/3, ls=":", c="grey"); ax[0].set_ylim(0, 1)
    ax[0].set_title("Pilot accuracy by Δstress (Δ=0 → correct answer is 'same')"); ax[0].set_xlabel("Δstress")
    acc_c = ok[ok.dstress > 0].groupby(["model", "cond"]).correct.mean().unstack(0)
    acc_c.plot.bar(ax=ax[1]); ax[1].axhline(0.5, ls=":", c="grey"); ax[1].set_ylim(0, 1)
    ax[1].set_title("Pilot accuracy by condition (Δ>0 only)"); ax[1].set_xlabel("")
    plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_pilot_accuracy.png", dpi=120); plt.show()
    print(ok.groupby("model").ans.value_counts(normalize=True).unstack().round(2))
    print("\nexample outputs:")
    for m, g in R.groupby("model"): print(f"--- {m}: {g.raw_text.iloc[0][:300]!r}")

ans             1     2
model                  
InternVL-4B  0.46  0.54
Q25VL-7B     0.38  0.62
Q3VL-2B      0.88  0.12
Q3VL-4B      0.35  0.65
Q3VL-8B      0.09  0.91
SmolVLM-2B   1.00   NaN

example outputs:
--- InternVL-4B: '{"answer": "1", "confidence": 4, "reason": "Image 1 shows nodes and edges more evenly distributed, with connections appearing closer for nodes that are few edges apart. This suggests a lower stress as the drawing more closely matches the graph-theoretic distances."}'
--- Q25VL-7B: '{"answer": "2", "confidence": 4, "reason": "Image 2 appears to have a more compact layout, suggesting that the distances between nodes that are few edges apart are closer together compared to Image 1."}'
--- Q3VL-2B: '```json\n{"answer": "1", "confidence": 5, "reason": "Image 1 shows a more compact arrangement where nodes that are close in the graph-theoretic sense are drawn close together, and nodes that are far apart are drawn far apart. This consistent alignment indicates a lower s

## 6. Manifest

In [9]:
shutil.rmtree(f"{OUTPUT_DIR}/__pycache__", ignore_errors=True)
lines = [f"N04 pilot · GPUs {N_GPU} · models ok {int(S.ok.fillna(False).sum())}/{len(S)} · runtime {(time.time()-T0)/60:.0f} min",
         f"primary analysis set: {int(main.primary.sum())} pairs · corr {json.dumps(CORR['primary'])}", ""]
for root, _, files in os.walk(OUTPUT_DIR):
    for f in sorted(files):
        p = os.path.join(root, f); lines.append(f"{os.path.relpath(p, OUTPUT_DIR):55s} {os.path.getsize(p)/1024:9.1f} KB")
open(f"{OUTPUT_DIR}/MANIFEST.txt", "w").write("\n".join(lines)); print("\n".join(lines))

N04 pilot · GPUs 2 · models ok 5/6 · runtime 78 min
primary analysis set: 1865 pairs · corr {"nu": {"pearson": -0.0229, "spearman": -0.0474}, "eld": {"pearson": 0.0071, "spearman": -0.0342}, "xr": {"pearson": 0.0061, "spearman": -0.0474}}

analysis_sets.parquet                                        31.9 KB
fig_pilot_accuracy.png                                      122.2 KB
pdcp_infer.py                                                14.9 KB
pdcp_utils.py                                                21.9 KB
pilot_pairs.csv                                              22.1 KB
pilot_summary.csv                                             1.4 KB
run_plan.json                                                 2.3 KB
results/pilot/InternVL-4B.jsonl                              87.4 KB
results/pilot/InternVL-4B.summary.json                        0.6 KB
results/pilot/Q25VL-7B.jsonl                                 71.1 KB
results/pilot/Q25VL-7B.summary.json                           0.4 KB
r